# Mp4LM Demo Notebook

A compact walkthrough of the current Mp4LM reference implementation.

- build a tiny `.mp4lm` bundle
- inspect the manifest and verify checksums
- selectively unpack one artifact
- optionally round-trip a PyTorch `state_dict`


In [ ]:
from pathlib import Path
import importlib.util
import struct
import sys
import tempfile

repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from mp4lm import BundleBuilder, Mp4LMBundle


In [ ]:
workspace = Path(tempfile.mkdtemp(prefix="mp4lm-demo-"))
bundle_path = workspace / "toy_model.mp4lm"
workspace


In [ ]:
tensor_bytes = struct.pack("<4f", 0.5, 1.5, 2.5, 3.5)

builder = BundleBuilder(
    model={"name": "toy-green-demo", "architecture": "decoder-only-transformer"},
    deterministic=True,
    chunk_size=8,
)
builder.add_tensor(
    "layers.0.weight",
    tensor_bytes,
    dtype="float32",
    shape=[2, 2],
    tensor_subtype="parameter",
)
builder.add_file("config.json", b'{"hidden_size": 2, "layers": 1}')
builder.write(bundle_path)
bundle_path


In [ ]:
bundle = Mp4LMBundle.open(bundle_path)
summary = {
    "spec": bundle.manifest()["spec"],
    "bundle_id": bundle.manifest()["bundle_id"],
    "artifact_names": [artifact["name"] for artifact in bundle.list_artifacts()],
    "chunk_count": len(bundle.manifest()["chunks"]),
}
summary


In [ ]:
bundle.verify()


In [ ]:
tensor = bundle.tensor("layers.0.weight")
selected_dir = workspace / "selected-output"
written = bundle.unpack(selected_dir, artifacts=["config.json"])

{
    "tensor_shape": tensor.shape,
    "tensor_dtype": tensor.dtype,
    "first_bytes": list(tensor.data[:8]),
    "written_files": [str(path.relative_to(workspace)) for path in written],
}


## Optional PyTorch Bridge

If `torch` is available, the next cell creates a tiny `state_dict`, packs it to `.mp4lm`, and restores it back into tensors.


In [ ]:
if importlib.util.find_spec("torch") is None:
    print("torch is not installed in this environment; skipping the PyTorch bridge demo.")
else:
    import torch
    from mp4lm import load_pytorch_state_dict, pack_pytorch_state_dict

    checkpoint_path = workspace / "toy_state_dict.pt"
    torch.save(
        {
            "embed.weight": torch.arange(0, 8, dtype=torch.float32).reshape(2, 4),
            "lm_head.weight": torch.arange(0, 4, dtype=torch.float32).reshape(1, 4),
        },
        checkpoint_path,
    )

    bridge_bundle_path = workspace / "toy_state_dict.mp4lm"
    pack_pytorch_state_dict(checkpoint_path, bridge_bundle_path, deterministic=True)
    roundtrip_state_dict = load_pytorch_state_dict(bridge_bundle_path)

    print(bridge_bundle_path)
    print(sorted(roundtrip_state_dict.keys()))
    print(roundtrip_state_dict["embed.weight"].shape)


## Next Ideas

- replace the toy bundle with a real checkpoint
- compare chunk sizing strategies
- wire the same flow into a remote byte-range loader
